In [1]:
import pandas as pd
import re
from tqdm import tqdm
import numpy as np
import gc
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize
import re
from tqdm import tqdm
from transformers import pipeline, AutoModelForMaskedLM, AutoTokenizer
from tqdm import tqdm
import torch




In [2]:
# Load transcripts
twitter_df = pd.read_csv("/kaggle/input/final-data/twitter.csv")

In [3]:
transcripts_list = twitter_df["text_clean"].tolist()
transcripts_list = [str(transcript) for transcript in tqdm(transcripts_list, desc="Tokenizing Transcripts")]
transcripts_list

Tokenizing Transcripts: 100%|██████████| 922146/922146 [00:00<00:00, 2829682.97it/s]


['Planungen Infektionsschutzgesetz sehen CoronaMaßnahmen fallen steigenden Infektions Hospitalisierungszahlen finde bedenklich Maskenpflicht Innenräumen bietet Einschränkungen guten Schutz httpstcogGlzhSm',
 'stattdessen hilft Energiegeld Menschen ausgezahlt tatsächlich ankommt brauchen',
 'Menschen Rechnungen Heizung Sprit Strom Tankrabatt Tankstellenbetreibern geholfen Menschen wenig Geld Tasche Angst nächsten Rechnung',
 'IngeHannemann setze Energiegeld pro Kopf Bürgerinnen Bürger ausgezahlt unabhängig Beschäftigungsverhältnis Rente',
 'brauchen schnell bessere soziale Entlastung Energiegeld zügig helfen Mehreinnahmen Besteuerung Energie Bürgerinnen zurückgeben Menschen wenig Geld entlasten httpstcovlElxNOSY',
 'Hermann Gröhe wohl mitbekommen Jobcenter Corona kaum sanktioniert Trotzdem bemühen Menschen Arbeit finden Sanktionen Jahres ausgesetzt werdennhttpstcoKFgBQpls',
 'Tarifrunde Sozial Erziehungsdienst Beschäftigten bessere Einkommen Arbeitsbedingungen verdient sorgen Menschen a

In [4]:
sentences = [transcript.split() for transcript in tqdm(transcripts_list, desc="Tokenizing Transcripts")]

Tokenizing Transcripts: 100%|██████████| 922146/922146 [00:03<00:00, 232382.52it/s]


In [5]:
print(f"Total sentences for training: {len(sentences)}")
print(f"Total tokens: {sum(len(sentence) for sentence in sentences)}")

print(f"Total sentences for training: {len(sentences)}")

model_path = '/kaggle/input/w2v-weights/word2vec_model.model'
model = Word2Vec.load(model_path)


# Update vocabulary with new sentences
print("Building Vocabulary with new data")
model.build_vocab(sentences, update=True)
print("Vocabulary updated with new data.")

# Training the model with progress bar
print("Start Training")

epochs = 10
total_sentences = len(sentences)

for epoch in range(epochs):
    print(f'Epoch {epoch+1}/{epochs}')
    pbar = tqdm(total=total_sentences, desc=f"Epoch {epoch+1}", unit="sentence")

    # Shuffle sentences for better training
    np.random.shuffle(sentences)
    
    # Define a batch size for training
    batch_size = 100000
    for i in range(0, total_sentences, batch_size):
        batch_sentences = sentences[i:i + batch_size]
        model.train(batch_sentences, total_examples=len(batch_sentences), epochs=1)
        pbar.update(len(batch_sentences))
    
    pbar.close()

print("Training completed.")

# Save the updated word vectors
model.wv.save_word2vec_format("word2vec_twitter.txt", binary=False)
model.save("word2vec_twitter.model")

print("Model and updated embeddings saved.")

Total sentences for training: 922146
Total tokens: 9415037
Total sentences for training: 922146
Building Vocabulary with new data
Vocabulary updated with new data.
Start Training
Epoch 1/10


Epoch 1: 100%|██████████| 922146/922146 [00:20<00:00, 43931.77sentence/s]


Epoch 2/10


Epoch 2: 100%|██████████| 922146/922146 [00:19<00:00, 48157.57sentence/s]


Epoch 3/10


...

Epoch 9/10


Epoch 9: 100%|██████████| 922146/922146 [00:18<00:00, 48950.02sentence/s]


Epoch 10/10


Epoch 10: 100%|██████████| 922146/922146 [00:19<00:00, 47963.96sentence/s]


Training completed.
Model and updated embeddings saved.
